# Section 3: Anomaly Detection for Cyber Threats

This self-contained notebook downloads and cleans an unmodified CSE-CIC-IDS2018 daily CSV, creates a benign-only training set, and compares Isolation Forest with a dense Autoencoder. It does not import project source files or call repository scripts.

In [ ]:
%pip install -q joblib matplotlib numpy pandas scikit-learn torch

In [ ]:
import copy
import hashlib
import json
import os
import random
import shutil
import urllib.request
from dataclasses import dataclass
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display
from sklearn.ensemble import IsolationForest
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay, average_precision_score, confusion_matrix,
    f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def find_local_root() -> Path:
    current = Path.cwd().resolve()
    return next((p for p in [current, *current.parents] if (p / "pyproject.toml").is_file()), current)

RUNNING_ON_COLAB = Path("/content").is_dir()
WORK_DIR = Path("/content/section_03_workspace") if RUNNING_ON_COLAB else find_local_root()
WORK_DIR.mkdir(parents=True, exist_ok=True); os.chdir(WORK_DIR)
config = {
    "seed": 42,
    "data_file": "data/raw/cic-ids2018/Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv",
    "processed_dir": "data/processed/section_03", "models_dir": "models/section_03", "results_dir": "reports/section_03",
    "max_benign_records": 120000, "max_anomaly_records": 18000,
    "benign_train_fraction": 0.6, "benign_validation_fraction": 0.2, "anomaly_validation_fraction": 0.5,
    "maximum_missing_fraction": 0.4,
    "isolation_forest": {"n_estimators": 200, "max_samples": 10000, "contamination": "auto"},
    "autoencoder": {"hidden_dimensions": [64, 32], "latent_dimension": 12, "dropout": 0.1, "batch_size": 1024, "epochs": 12, "learning_rate": 0.001, "weight_decay": 0.00001, "early_stopping_patience": 3},
}
random.seed(config["seed"]); np.random.seed(config["seed"])
print("Environment:", "Google Colab" if RUNNING_ON_COLAB else "Local")
print("Working directory:", WORK_DIR)
print("CUDA available:", torch.cuda.is_available())

## 1. Download and verify the unclean CIC-IDS2018 CSV

In [ ]:
CIC_URL = "https://cse-cic-ids2018.s3.amazonaws.com/Processed%20Traffic%20Data%20for%20ML%20Algorithms/Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv"
CIC_BYTES = 107842858
CIC_SHA256 = "b0534c5d7d8b41e03df71c6966c995d116a8ed28e61f377c8b14cdf5d28f4edf"

def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def validate_cic_file(path: Path) -> None:
    if path.stat().st_size != CIC_BYTES or file_sha256(path) != CIC_SHA256:
        raise ValueError(f"Integrity check failed for {path}")

def download_cic_ids2018(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.is_file():
        temporary = path.with_suffix(path.suffix + ".part")
        request = urllib.request.Request(CIC_URL, headers={"User-Agent": "COMP70049-assignment/1.0"})
        print("Downloading approximately 103 MiB...")
        with urllib.request.urlopen(request, timeout=120) as response, temporary.open("wb") as output:
            while chunk := response.read(1024 * 1024): output.write(chunk)
        validate_cic_file(temporary); temporary.replace(path)
    validate_cic_file(path)
    print("Verified:", path)

download_cic_ids2018(Path(config["data_file"]))

## 2. Audit, clean, and split the raw data

The cleaning logic removes duplicate and embedded-header rows, replaces infinity, coerces malformed numeric cells, derives time features, and retains statistical outliers.

In [ ]:
TRAFFIC_FEATURES = (
    "Dst Port", "Protocol", "Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts", "TotLen Fwd Pkts", "TotLen Bwd Pkts",
    "Fwd Pkt Len Max", "Fwd Pkt Len Min", "Fwd Pkt Len Mean", "Fwd Pkt Len Std", "Bwd Pkt Len Max", "Bwd Pkt Len Min", "Bwd Pkt Len Mean", "Bwd Pkt Len Std",
    "Flow Byts/s", "Flow Pkts/s", "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
    "Fwd IAT Tot", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max", "Fwd IAT Min", "Bwd IAT Tot", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min",
    "Fwd Pkts/s", "Bwd Pkts/s", "Pkt Len Min", "Pkt Len Max", "Pkt Len Mean", "Pkt Len Std", "Pkt Len Var",
    "FIN Flag Cnt", "SYN Flag Cnt", "RST Flag Cnt", "PSH Flag Cnt", "ACK Flag Cnt", "URG Flag Cnt", "Down/Up Ratio", "Pkt Size Avg",
    "Fwd Seg Size Avg", "Bwd Seg Size Avg", "Subflow Fwd Pkts", "Subflow Fwd Byts", "Subflow Bwd Pkts", "Subflow Bwd Byts",
    "Init Fwd Win Byts", "Init Bwd Win Byts", "Fwd Act Data Pkts", "Fwd Seg Size Min",
    "Active Mean", "Active Std", "Active Max", "Active Min", "Idle Mean", "Idle Std", "Idle Max", "Idle Min", "hour", "day_of_week",
)

@dataclass(frozen=True)
class CleanedDataset:
    frame: pd.DataFrame
    feature_columns: tuple[str, ...]
    raw_profile: dict[str, object]
    cleaning_report: dict[str, object]

@dataclass(frozen=True)
class AnomalyPartitions:
    train: pd.DataFrame
    validation: pd.DataFrame
    test: pd.DataFrame

def clean_cic_frame(raw: pd.DataFrame, maximum_missing_fraction: float) -> CleanedDataset:
    raw_profile = {"records": int(len(raw)), "columns": int(raw.shape[1]), "duplicate_records": int(raw.duplicated().sum()), "native_missing_values": int(raw.isna().sum().sum())}
    frame = raw.copy(); frame.columns = [str(c).replace("\ufeff", "").strip() for c in frame.columns]
    frame = frame.drop(columns=[c for c in frame.columns if c.lower().startswith("unnamed:")])
    labels = frame["Label"].astype("string").str.strip()
    missing_label = labels.isna() | labels.eq(""); embedded_header = labels.str.casefold().eq("label")
    frame = frame.loc[~(missing_label | embedded_header)].copy()
    duplicate_records = int(frame.duplicated().sum()); frame = frame.drop_duplicates().reset_index(drop=True)
    labels = frame["Label"].astype("string").str.strip()
    timestamps = pd.to_datetime(frame["Timestamp"].astype("string").str.strip(), errors="coerce", format="mixed", dayfirst=True)
    frame["hour"] = timestamps.dt.hour.astype(float); frame["day_of_week"] = timestamps.dt.dayofweek.astype(float)
    available = [column for column in TRAFFIC_FEATURES if column in frame.columns]
    numeric, infinities, coercions = pd.DataFrame(index=frame.index), 0, 0
    for column in available:
        original = frame[column]; converted = pd.to_numeric(original, errors="coerce")
        coercions += int((original.notna() & original.astype("string").str.strip().ne("") & converted.isna()).sum())
        infinities += int(np.isinf(converted.to_numpy(dtype=float, copy=True)).sum())
        numeric[column] = converted.replace([np.inf, -np.inf], np.nan)
    dropped = numeric.isna().mean(); dropped = sorted(dropped[dropped > maximum_missing_fraction].index.tolist()); numeric = numeric.drop(columns=dropped)
    cleaned = numeric.copy(); cleaned["attack_label"] = labels.to_numpy(dtype=str); cleaned["label"] = (~labels.str.casefold().eq("benign")).astype(int).to_numpy()
    report = {
        "records_after_cleaning": int(len(cleaned)), "rows_without_label_removed": int(missing_label.sum()),
        "embedded_header_rows_removed": int(embedded_header.sum()), "duplicate_records_removed": duplicate_records,
        "infinite_values_replaced_with_missing": infinities, "non_numeric_values_coerced_to_missing": coercions,
        "invalid_timestamps": int(timestamps.isna().sum()), "features_dropped_for_missingness": dropped,
        "remaining_missing_values": int(numeric.isna().sum().sum()),
        "class_counts": {"benign": int((cleaned["label"] == 0).sum()), "anomaly": int((cleaned["label"] == 1).sum())},
    }
    return CleanedDataset(cleaned, tuple(numeric.columns), raw_profile, report)

def sample_records(frame: pd.DataFrame, maximum: int, seed: int) -> pd.DataFrame:
    return frame.sample(n=min(len(frame), maximum), random_state=seed).reset_index(drop=True)

def create_partitions(frame: pd.DataFrame, cfg: dict[str, object]) -> AnomalyPartitions:
    seed = int(cfg["seed"]); benign = sample_records(frame[frame["label"] == 0], int(cfg["max_benign_records"]), seed); anomalies = sample_records(frame[frame["label"] == 1], int(cfg["max_anomaly_records"]), seed + 1)
    train_end = int(len(benign) * float(cfg["benign_train_fraction"])); validation_end = train_end + int(len(benign) * float(cfg["benign_validation_fraction"])); anomaly_end = int(len(anomalies) * float(cfg["anomaly_validation_fraction"]))
    train = benign.iloc[:train_end].copy()
    validation = pd.concat([benign.iloc[train_end:validation_end], anomalies.iloc[:anomaly_end]], ignore_index=True).sample(frac=1, random_state=seed + 2).reset_index(drop=True)
    test = pd.concat([benign.iloc[validation_end:], anomalies.iloc[anomaly_end:]], ignore_index=True).sample(frac=1, random_state=seed + 3).reset_index(drop=True)
    return AnomalyPartitions(train, validation, test)

raw = pd.read_csv(config["data_file"], low_memory=False)
cleaned = clean_cic_frame(raw, float(config["maximum_missing_fraction"]))
partitions = create_partitions(cleaned.frame, config)
display(pd.Series({**cleaned.raw_profile, **cleaned.cleaning_report}, name="value").to_frame())
display(pd.DataFrame([
    {"split": name, "records": len(frame), "benign": int((frame["label"] == 0).sum()), "anomaly": int((frame["label"] == 1).sum())}
    for name, frame in (("train", partitions.train), ("validation", partitions.validation), ("test", partitions.test))
]))

## 3. Preprocessing, threshold selection, metrics, and plots

In [ ]:
def build_preprocessor() -> Pipeline:
    return Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True)), ("variance", VarianceThreshold()), ("scaler", StandardScaler())])

def transformed_names(pipeline: Pipeline, columns: tuple[str, ...]) -> list[str]:
    names = pipeline.named_steps["imputer"].get_feature_names_out(columns)
    return [str(name) for name in names[pipeline.named_steps["variance"].get_support()]]

def select_f1_threshold(labels: np.ndarray, scores: np.ndarray) -> float:
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    denominator = precision[:-1] + recall[:-1]
    values = np.divide(2 * precision[:-1] * recall[:-1], denominator, out=np.zeros_like(denominator), where=denominator > 0)
    return float(thresholds[int(np.nanargmax(values))])

def anomaly_metrics(labels: np.ndarray, scores: np.ndarray, threshold: float) -> dict[str, object]:
    predictions = (scores >= threshold).astype(int); tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold), "true_positive_rate": float(tp / (tp + fn)), "false_positive_rate": float(fp / (fp + tn)),
        "precision": float(precision_score(labels, predictions, zero_division=0)), "recall": float(recall_score(labels, predictions, zero_division=0)),
        "f1": float(f1_score(labels, predictions, zero_division=0)), "average_precision": float(average_precision_score(labels, scores)),
        "roc_auc": float(roc_auc_score(labels, scores)), "confusion_matrix": [[int(tn), int(fp)], [int(fn), int(tp)]],
    }

def save_anomaly_artifacts(slug: str, name: str, labels: np.ndarray, scores: np.ndarray, metrics: dict[str, object], results_dir: Path) -> None:
    metrics_dir, figures_dir = results_dir / "metrics", results_dir / "figures"; metrics_dir.mkdir(parents=True, exist_ok=True); figures_dir.mkdir(parents=True, exist_ok=True)
    (metrics_dir / f"{slug}.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    predictions = (scores >= float(metrics["threshold"])).astype(int)
    display_plot = ConfusionMatrixDisplay(confusion_matrix(labels, predictions, labels=[0, 1]), display_labels=["Benign", "Anomaly"]); display_plot.plot(cmap="Blues", colorbar=False); plt.title(f"{name} confusion matrix"); plt.tight_layout(); plt.savefig(figures_dir / f"{slug}-confusion-matrix.png", dpi=180); plt.close()
    precision, recall, _ = precision_recall_curve(labels, scores); plt.figure(figsize=(6.4, 4.5)); plt.plot(recall, precision); plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(f"{name} precision-recall curve"); plt.grid(alpha=0.25); plt.tight_layout(); plt.savefig(figures_dir / f"{slug}-precision-recall-curve.png", dpi=180); plt.close()
    plt.figure(figsize=(6.4, 4.5)); plt.hist(scores[labels == 0], bins=60, alpha=0.65, label="Benign"); plt.hist(scores[labels == 1], bins=60, alpha=0.65, label="Anomaly"); plt.axvline(float(metrics["threshold"]), color="black", linestyle="--"); plt.legend(); plt.title(f"{name} score distribution"); plt.tight_layout(); plt.savefig(figures_dir / f"{slug}-score-distribution.png", dpi=180); plt.close()

preprocessor = build_preprocessor()
train_features = np.asarray(preprocessor.fit_transform(partitions.train[list(cleaned.feature_columns)]), dtype=np.float32)
validation_features = np.asarray(preprocessor.transform(partitions.validation[list(cleaned.feature_columns)]), dtype=np.float32)
test_features = np.asarray(preprocessor.transform(partitions.test[list(cleaned.feature_columns)]), dtype=np.float32)
feature_names = transformed_names(preprocessor, cleaned.feature_columns)
validation_labels = partitions.validation["label"].to_numpy(dtype=np.int64); test_labels = partitions.test["label"].to_numpy(dtype=np.int64)
models_dir, results_dir = Path(config["models_dir"]), Path(config["results_dir"]); models_dir.mkdir(parents=True, exist_ok=True)

## 4. Isolation Forest

In [ ]:
def train_isolation_forest(features: np.ndarray, model_config: dict[str, object], seed: int) -> IsolationForest:
    model = IsolationForest(n_estimators=int(model_config["n_estimators"]), max_samples=model_config["max_samples"], contamination=model_config["contamination"], random_state=seed, n_jobs=-1)
    model.fit(features); return model

isolation_forest = train_isolation_forest(train_features, config["isolation_forest"], config["seed"])
validation_scores = -isolation_forest.score_samples(validation_features); isolation_threshold = select_f1_threshold(validation_labels, validation_scores)
isolation_scores = -isolation_forest.score_samples(test_features); isolation_metrics = anomaly_metrics(test_labels, isolation_scores, isolation_threshold)
isolation_metrics["validation_threshold_method"] = "maximum F1"
save_anomaly_artifacts("isolation-forest", "Isolation Forest", test_labels, isolation_scores, isolation_metrics, results_dir)
joblib.dump({"model": isolation_forest, "preprocessor": preprocessor, "threshold": isolation_threshold, "feature_names": feature_names}, models_dir / "isolation-forest.joblib")
display(pd.Series(isolation_metrics).drop("confusion_matrix").to_frame("value"))

## 5. Dense Autoencoder

In [ ]:
class DenseAutoencoder(nn.Module):
    def __init__(self, input_dimension: int, hidden_dimensions: list[int], latent_dimension: int, dropout: float) -> None:
        super().__init__(); encoder, previous = [], input_dimension
        for width in hidden_dimensions: encoder.extend([nn.Linear(previous, width), nn.ReLU(), nn.Dropout(dropout)]); previous = width
        encoder.append(nn.Linear(previous, latent_dimension)); self.encoder = nn.Sequential(*encoder)
        decoder, previous = [], latent_dimension
        for width in reversed(hidden_dimensions): decoder.extend([nn.Linear(previous, width), nn.ReLU(), nn.Dropout(dropout)]); previous = width
        decoder.append(nn.Linear(previous, input_dimension)); self.decoder = nn.Sequential(*decoder)
    def forward(self, values: torch.Tensor) -> torch.Tensor: return self.decoder(self.encoder(values))

def feature_loader(features: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    return DataLoader(TensorDataset(torch.from_numpy(np.asarray(features, dtype=np.float32))), batch_size=batch_size, shuffle=shuffle)

def reconstruction_scores(model: nn.Module, features: np.ndarray, batch_size: int, device: torch.device) -> np.ndarray:
    model.eval(); scores = []
    with torch.no_grad():
        for (batch,) in feature_loader(features, batch_size, False):
            batch = batch.to(device); scores.append(torch.mean((model(batch) - batch) ** 2, dim=1).cpu().numpy())
    return np.concatenate(scores)

def train_autoencoder(train_features, validation_features, validation_labels, model_config, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = DenseAutoencoder(train_features.shape[1], [int(v) for v in model_config["hidden_dimensions"]], int(model_config["latent_dimension"]), float(model_config["dropout"])).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=float(model_config["learning_rate"]), weight_decay=float(model_config["weight_decay"])); loss_function = nn.MSELoss(); batch_size = int(model_config["batch_size"])
    train_loader = feature_loader(train_features, batch_size, True); normal_validation = validation_features[validation_labels == 0]
    history, best_state, best_loss, stale = [], copy.deepcopy(model.state_dict()), float("inf"), 0
    for epoch in range(1, int(model_config["epochs"]) + 1):
        model.train(); total = 0.0
        for (batch,) in train_loader:
            batch = batch.to(device); optimizer.zero_grad(set_to_none=True); loss = loss_function(model(batch), batch); loss.backward(); optimizer.step(); total += float(loss.item()) * len(batch)
        model.eval(); validation_total = 0.0
        with torch.no_grad():
            for (batch,) in feature_loader(normal_validation, batch_size, False): batch = batch.to(device); validation_total += float(loss_function(model(batch), batch).item()) * len(batch)
        train_loss, validation_loss = total / len(train_features), validation_total / len(normal_validation); history.append({"epoch": epoch, "train_loss": train_loss, "normal_validation_loss": validation_loss})
        print(f"Epoch {epoch}: train_loss={train_loss:.4f}, validation_loss={validation_loss:.4f}")
        if validation_loss < best_loss - 1e-7: best_loss, best_state, stale = validation_loss, copy.deepcopy(model.state_dict()), 0
        else:
            stale += 1
            if stale >= int(model_config["early_stopping_patience"]): break
    model.load_state_dict(best_state); return model, device, history, best_loss

AUTOENCODER_EPOCHS_OVERRIDE = None  # Change to 1 for a quick smoke run.
autoencoder_config = config["autoencoder"].copy()
if AUTOENCODER_EPOCHS_OVERRIDE is not None: autoencoder_config["epochs"] = AUTOENCODER_EPOCHS_OVERRIDE
autoencoder, device, history, best_loss = train_autoencoder(train_features, validation_features, validation_labels, autoencoder_config, config["seed"])
validation_scores = reconstruction_scores(autoencoder, validation_features, int(autoencoder_config["batch_size"]), device); autoencoder_threshold = select_f1_threshold(validation_labels, validation_scores)
autoencoder_scores = reconstruction_scores(autoencoder, test_features, int(autoencoder_config["batch_size"]), device); autoencoder_metrics = anomaly_metrics(test_labels, autoencoder_scores, autoencoder_threshold)
autoencoder_metrics.update({"validation_threshold_method": "maximum F1", "epochs_completed": len(history), "best_normal_validation_loss": best_loss, "device": str(device), "training_history": history})
save_anomaly_artifacts("autoencoder", "Autoencoder", test_labels, autoencoder_scores, autoencoder_metrics, results_dir)
torch.save({"state_dict": autoencoder.state_dict(), "threshold": autoencoder_threshold, "feature_names": feature_names, "config": autoencoder_config}, models_dir / "autoencoder.pt")
epochs = [row["epoch"] for row in history]; plt.figure(figsize=(6.4, 4.5)); plt.plot(epochs, [row["train_loss"] for row in history], label="Train"); plt.plot(epochs, [row["normal_validation_loss"] for row in history], label="Normal validation"); plt.legend(); plt.grid(alpha=0.25); plt.title("Autoencoder training history"); plt.tight_layout(); plt.savefig(results_dir / "figures/autoencoder-training-history.png", dpi=180); plt.close()
display(pd.Series(autoencoder_metrics).drop(["confusion_matrix", "training_history"]).to_frame("value"))

## 6. Compare and export

High true-positive rate is not useful when false-positive rate is also high. Interpret the threshold trade-off, precision-recall curve, and score ranking together.

In [ ]:
metric_names = ["true_positive_rate", "false_positive_rate", "precision", "f1", "average_precision", "roc_auc"]
comparison = pd.DataFrame([
    {"model": "Isolation Forest", **{name: isolation_metrics[name] for name in metric_names}},
    {"model": "Autoencoder", **{name: autoencoder_metrics[name] for name in metric_names}},
])
results_dir.mkdir(parents=True, exist_ok=True); comparison.to_csv(results_dir / "model-comparison.csv", index=False)
processed_dir = Path(config["processed_dir"]); processed_dir.mkdir(parents=True, exist_ok=True)
(processed_dir / "data-quality-report.json").write_text(json.dumps({"raw_profile": cleaned.raw_profile, "cleaning_report": cleaned.cleaning_report}, indent=2), encoding="utf-8")
(processed_dir / "selected-features.json").write_text(json.dumps(feature_names, indent=2), encoding="utf-8")
display(comparison.style.format(precision=4))
if RUNNING_ON_COLAB:
    export = WORK_DIR / "section_03_export"
    if export.exists(): shutil.rmtree(export)
    shutil.copytree(results_dir, export / "reports"); shutil.copytree(models_dir, export / "models")
    print("Download:", shutil.make_archive("/content/section_03_results", "zip", root_dir=export))

## Limitations

The selected day contains infiltration traffic from a controlled 2018 testbed. Labels are used only for validation threshold selection and final evaluation, but dataset artefacts, the random split, and the difference between anomalies and confirmed attacks prevent deployment claims.